In [2]:
import pandas as pd

In [8]:
# Load the raw datasets
df_customers = pd.read_csv('archive/olist_customers_dataset.csv')
df_products = pd.read_csv('archive/olist_products_dataset.csv')
df_sellers = pd.read_csv('archive/olist_sellers_dataset.csv')
df_orders = pd.read_csv('archive/olist_orders_dataset.csv')
df_items = pd.read_csv('archive/olist_order_items_dataset.csv')
df_geo = pd.read_csv('archive/olist_geolocation_dataset.csv')
df_translation = pd.read_csv('archive/product_category_name_translation.csv')

# Convert timestamps immediately
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])

In [9]:
# Create Dim Geolocation
dim_geolocation = df_geo.copy()
# Keep only the first latitude/longitude for each zip code to avoid duplicates
dim_geolocation = dim_geolocation.drop_duplicates(subset=['geolocation_zip_code_prefix']).reset_index(drop=True)

# Generate Primary Key
dim_geolocation.insert(0, 'geolocation_key', dim_geolocation.index + 1)

# Ensure column names match your diagram
dim_geolocation = dim_geolocation[['geolocation_key', 'geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']]

In [10]:
# Merge products with the English translation
dim_products = df_products.merge(df_translation, on='product_category_name', how='left')

# Drop the Portuguese column and rename the English one to match your diagram
dim_products = dim_products.rename(columns={'product_category_name_english': 'product_category_name_english'})
dim_products = dim_products[['product_id', 'product_category_name_english', 'product_weight_g', 'product_length_cm', 'product_height_cm']].copy()

# Deduplicate and generate Primary Key
dim_products = dim_products.drop_duplicates(subset=['product_id']).reset_index(drop=True)
dim_products.insert(0, 'product_key', dim_products.index + 1)

In [13]:
# 1. Dim Customers
dim_customers = df_customers[['customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']].copy()
dim_customers = dim_customers.drop_duplicates(subset=['customer_unique_id']).reset_index(drop=True)

# Merge with geolocation to get the FK
dim_customers = dim_customers.merge(dim_geolocation[['geolocation_zip_code_prefix', 'geolocation_key']], 
                                    left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')

# Generate PK and order columns
dim_customers.insert(0, 'customer_key', dim_customers.index + 1)
dim_customers = dim_customers[['customer_key', 'geolocation_key', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']]

# 2. Dim Sellers
dim_sellers = df_sellers[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']].copy()
dim_sellers = dim_sellers.drop_duplicates(subset=['seller_id']).reset_index(drop=True)

# Merge with geolocation to get the FK
dim_sellers = dim_sellers.merge(dim_geolocation[['geolocation_zip_code_prefix', 'geolocation_key']], 
                                left_on='seller_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')

# Generate PK and order columns
dim_sellers.insert(0, 'seller_key', dim_sellers.index + 1)
dim_sellers = dim_sellers[['seller_key', 'geolocation_key', 'seller_id', 'seller_city', 'seller_state']]

In [14]:
# Extract unique dates
dim_date = pd.DataFrame({'full_date': df_orders['order_purchase_timestamp'].dt.date})
dim_date = dim_date.drop_duplicates().dropna().reset_index(drop=True)

# Create keys and parts
dim_date.insert(0, 'date_key', dim_date['full_date'].astype(str).str.replace('-', '').astype(int))
dim_date['month'] = pd.to_datetime(dim_date['full_date']).dt.month
dim_date['year'] = pd.to_datetime(dim_date['full_date']).dt.year
dim_date['day_of_week'] = pd.to_datetime(dim_date['full_date']).dt.dayofweek

In [15]:
# Start with order items and merge necessary IDs
fact_df = df_items.merge(df_orders[['order_id', 'customer_id', 'order_purchase_timestamp']], on='order_id', how='left')
fact_df = fact_df.merge(df_customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left')

# Create the date key
fact_df['order_purchase_date_key'] = fact_df['order_purchase_timestamp'].dt.strftime('%Y%m%d').fillna(0).astype(int)

# Merge with Dimensions to get Keys
fact_df = fact_df.merge(dim_customers[['customer_unique_id', 'customer_key']], on='customer_unique_id', how='left')
fact_df = fact_df.merge(dim_products[['product_id', 'product_key']], on='product_id', how='left')
fact_df = fact_df.merge(dim_sellers[['seller_id', 'seller_key']], on='seller_id', how='left')

# Final formatting
fact_order_items = fact_df[['order_id', 'order_item_id', 'customer_key', 'product_key', 'seller_key', 'order_purchase_date_key', 'price', 'freight_value']].copy()
fact_order_items = fact_order_items.rename(columns={'order_item_id': 'order_item_sequence_id'})
fact_order_items.insert(0, 'fact_key', fact_order_items.index + 1)

In [16]:
revenue_df = fact_order_items.merge(dim_products, on='product_key', how='inner')
top_categories = revenue_df.groupby('product_category_name_english')['price'].sum().sort_values(ascending=False).head(5)
print("\nTop 5 Categories by Revenue:\n", top_categories)


Top 5 Categories by Revenue:
 product_category_name_english
health_beauty            1258681.34
watches_gifts            1205005.68
bed_bath_table           1036988.68
sports_leisure            988048.97
computers_accessories     911954.32
Name: price, dtype: float64


In [18]:
fact_order_items

,fact_key,order_id,order_item_sequence_id,customer_key,product_key,seller_key,order_purchase_date_key,price,freight_value
0,1,00010242fe8c5a6d1ba2dd792cb16214,1,64073,25866,514,20170913,58.90,13.29
1,2,00018f77f2f0320c557190d7a144bdd3,1,33838,27231,472,20170426,239.90,19.93
2,3,000229ec398224ef6ca0657da4fc703e,1,34511,22625,1825,20180114,199.00,17.87
3,4,00024acbcdf0a6daa1e931b038114c75,1,50803,15404,2024,20180808,12.99,12.79
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,1,7582,8863,1598,20170204,199.90,18.14
...,...,...,...,...,...,...,...,...,...
112645,112646,fffc94f6ce00a00581880bf54a75a037,1,85643,4738,2324,20180423,299.99,43.41
112646,112647,fffcd46ef2263f404302a634eb57f7eb,1,26405,9256,1776,20180714,350.00,36.53
112647,112648,fffce4705a9662cd70adb13d4a31832d,1,1506,24016,602,20171023,99.90,16.95
112648,112649,fffe18544ffabc95dfada21779c9644f,1,55521,22356,2877,20170814,55.99,8.72
